# Three-site eigenvalues and paper comparisons

Reproduce the numerical references for Eqs. (9–12), Figs. 3–4, and Sec. V B. The saved coefficients below are the published, rounded Eq. (12), not a reconstructed original checkpoint. A complete fresh training command appears at the end.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments.paper import ROOT
print('Repository:', ROOT)

In [ ]:
from experiments.paper import (exact_three, perturbation_three, published_predictions,
                              grid_three, basis, make_dataset, metrics)
local = grid_three(.2, 101)
local_error = metrics(perturbation_three(local), exact_three(local))
print('Local second-order perturbation:', local_error)
assert abs(local_error['rmse'] - 4.28e-5) < 5e-8
assert abs(local_error['max_abs_error'] - 2.64e-4) < 5e-7

The leading ordered splitting is radial in $(r_1,r_2)$, and is nonanalytic at the origin. The ratios containing $1/(r_1^2+r_2^2)$ are assigned their continuous limit of zero there. Along the uniform line the signed perturbative branches cross; the energy-ordered spectrum has a cusp.

In [ ]:
r = np.linspace(-2, 2, 401)
x = np.column_stack((r,r))
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(r, exact_three(x))
ax.plot(r, published_predictions(abs(x)), 'k--', alpha=.5)
ax.set(xlabel='r1 = r2', ylabel='E/J', title='Exact branches and reflected rounded Eq. (12)')
plt.show()

## Global and local errors
Use the original notebook's NumPy seed and separate coordinate draws. OLS validation coefficients are fitted on the training subset; the complete-sample fit is separately labeled. Globally fitted models are evaluated on the small local grid without refitting.

In [ ]:
config = json.loads((ROOT/'configs/paper/three_site.json').read_text())
x, y, train, validation = make_dataset(config)
quadratic = basis(x); radial = basis(x, True)
quad_coeff = np.linalg.lstsq(quadratic[train], y[train], rcond=None)[0]
radial_coeff = np.linalg.lstsq(radial[train], y[train], rcond=None)[0]
full_coeff = np.linalg.lstsq(quadratic, y, rcond=None)[0]
global_grid = grid_three()
comparisons = {
    'OLS quadratic, held-out': metrics(quadratic[validation]@quad_coeff, y[validation]),
    'OLS radial, held-out': metrics(radial[validation]@radial_coeff, y[validation]),
    'OLS quadratic, complete-sample fit': metrics(quadratic@full_coeff, y),
    'OLS quadratic, local grid': metrics(basis(local)@quad_coeff, exact_three(local)),
    'OLS radial, local grid': metrics(basis(local, True)@radial_coeff, exact_three(local)),
    'Perturbation, local grid': local_error,
    'Rounded Eq. (12), 121x121 grid': metrics(published_predictions(global_grid), exact_three(global_grid)),
}
pd.DataFrame(comparisons).T[['rmse','max_abs_error']]

In [ ]:
a = np.linspace(-2,2,81); aa,bb=np.meshgrid(a,a)
points=np.column_stack((aa.ravel(),bb.ravel()))
exact=exact_three(points); predicted=published_predictions(abs(points))
fig,axes=plt.subplots(4,2,figsize=(9,12),layout='constrained')
for k in range(4):
    lo=min(exact[:,k].min(),predicted[:,k].min()); hi=max(exact[:,k].max(),predicted[:,k].max())
    for j,(values,label) in enumerate([(exact,'Exact'),(predicted,'Rounded Eq. (12)')]):
        im=axes[k,j].imshow(values[:,k].reshape(aa.shape),origin='lower',extent=(-2,2,-2,2),vmin=lo,vmax=hi)
        axes[k,j].set(title=f'{label}, branch {k+1}',xlabel='r1',ylabel='r2')
        fig.colorbar(im,ax=axes[k,j])
plt.show()

## Train and export

From the repository root:
```bash
python -m experiments.paper --config configs/paper/three_site.json
python -m experiments.paper --config configs/paper/three_site.json --train
```
The first exports these numerical comparisons and figures. The second trains `(2,16,4)` for 1000 epochs and writes a fresh checkpoint, history, full-precision formulas and processed formulas. The original 1.79e-2 neural error cannot be reconstructed without its checkpoint. See [the result map](../docs/reproducing_the_paper.md) for exact scope and [04](04_training_walkthrough.ipynb) for a short training walkthrough.